# Task-Conditioned Evaluation of Quantized Small LLMs (BTP GP159)
### Delhi Technological University | B.Tech Major Project (Session 2026–27)
**Team:** Ekansh Agarwal (`23/CS/149`), Faqre Alam (`23/CS/150`), Guneet Toppo (`23/CS/160`)
**Supervisor:** Ms. Gull Kaur, Assistant Professor, Department of CSE, DTU

---
### Kaggle T4 Execution Instructions:
1. **Accelerator:** Under the right panel (*Session options*), set **Accelerator** to **GPU T4 x1** (16 GB VRAM).
2. **Internet:** Turn **Internet ON** in the right panel under *Session options*.
3. **Hugging Face Token:** Required for downloading gated models (`Llama-3.2-1B-Instruct`). See the step-by-step guide in Step 3 below on how to obtain and add your `HF_TOKEN` via Kaggle Secrets.

In [ ]:
# 1. Verify NVIDIA T4 GPU Environment
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")

In [ ]:
# 1b. Set up repository in Kaggle workspace
import os
if not os.path.exists("run_experiments.py"):
    if os.path.exists("final-year-btp"):
        %cd final-year-btp
    else:
        !git clone https://github.com/itsalam149/final-year-btp.git
        %cd final-year-btp

# Safely pull latest code updates from GitHub without wiping local results
if os.path.exists(".git"):
    !git pull origin main


In [ ]:
# 2. Install Dependencies for Evaluation
!pip install -q "datasets==2.21.0" "huggingface-hub<0.29.0" lm-eval==0.4.3
!pip install -q accelerate bitsandbytes optimum langdetect immutabledict gptqmodel tabulate


---
## 🔑 Hugging Face API / Access Token Guide

### Why is a Hugging Face Token Required?
- **Gated Models:** Meta's `Llama-3.2-1B-Instruct` is a gated repository on Hugging Face. Access requires agreeing to Meta's community license terms and authenticating with an API token.
- **Rate Limit Protection:** An authenticated token provides significantly higher Hugging Face Hub download rate limits, preventing HTTP 429 errors when downloading benchmark datasets (`mmlu`, `gsm8k`, `ifeval`, etc.).
- *Note:* `Qwen2.5-0.5B-Instruct` is open and will evaluate without a token, but the token is essential to execute the Llama-3.2 test suite and complete the cross-architecture evaluation.

---
### Step-by-Step: How to Get Your Hugging Face API Token
1. **Create / Log in to Hugging Face:**
   - Go to [huggingface.co](https://huggingface.co/) and sign in to your account (or create one for free).
2. **Accept the Llama 3.2 License Agreement:**
   - Visit the official repository: [meta-llama/Llama-3.2-1B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct).
   - Fill in the contact information / affiliation (e.g., student / Delhi Technological University), review Meta's license, and click **Submit / Accept**.
   - *Access is granted automatically within seconds.*
3. **Generate a Read Token:**
   - Click your profile icon in the top right corner ➔ **Settings** ➔ **Access Tokens** (or visit [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)).
   - Click **"Create new token"** (or **"New token"**).
   - Under **Token type**, select **"Read"** (or fine-grained with read access to public/gated repos).
   - Set a name such as `kaggle-btp`.
   - Click **"Create token"** and copy the string (it starts with `hf_...`).

---
### How to Apply the Token in Kaggle:

#### Method 1: Kaggle Secrets (Recommended & Persistent)
1. In the Kaggle notebook top navigation bar, click **Add-ons** ➔ **Secrets**.
2. Click **+ Add a new secret**.
3. In the **Label** field, enter: `HF_TOKEN` *(exact uppercase)*.
4. In the **Value** field, paste your copied token (`hf_...`).
5. Check the checkbox next to `HF_TOKEN` so it is attached to this notebook session.
6. Click **Save**.
*Advantage:* Your secret is securely encrypted and never leaked in notebook outputs or git commits.

#### Method 2: Interactive Prompt (Runtime Fallback)
If you haven't added the secret via Kaggle Add-ons, running the cell below will automatically display a hidden input prompt (`getpass`) where you can paste your token directly into runtime memory.

In [ ]:
# 3. Hugging Face Login (for gated Llama-3.2-1B and dataset Hub access)
import os
import getpass
from huggingface_hub import login, HfApi

hf_token = os.environ.get("HF_TOKEN")

# Attempt 1: Fetch from Kaggle Secrets (Add-ons -> Secrets -> HF_TOKEN)
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        pass

# Attempt 2: Interactive prompt fallback if running in notebook without Secrets attached
if not hf_token:
    try:
        interactive_token = getpass.getpass("Enter Hugging Face Token (hf_...) [Press Enter to skip]: ").strip()
        if interactive_token:
            hf_token = interactive_token
    except Exception:
        pass

# Authenticate and Verify identity
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token, add_to_git_credential=False)
    try:
        user_info = HfApi().whoami(token=hf_token)
        username = user_info.get("name", "User")
        print(f"✅ Hugging Face Authentication Successful! Logged in as: @{username}")
    except Exception:
        print("✅ Hugging Face Authentication Successful!")
else:
    print("⚠️ No HF_TOKEN provided.")
    print("ℹ️ Note: Qwen2.5 runs will execute freely without authentication.")
    print("ℹ️ To evaluate Llama-3.2-1B, add HF_TOKEN via Kaggle Secrets (Add-ons -> Secrets) or enter it in the prompt above.")


In [ ]:
# 4. Verify Grid and Dry-Run
!python run_experiments.py --dry-run

In [ ]:
# 5. Phase B+ Audit: Checkpoint Provenance Verification
!python scripts/audit_checkpoints.py --update-yaml

In [ ]:
# 6. Phase B: Execute FP16 Baselines (Both Models)
!python run_experiments.py --variant fp16 --skip-completed


In [ ]:
# 7. Phase C: Execute Core Quantization Runs (8-bit and 4-bit) on Standard Benchmarks
!python run_experiments.py --tasks wikitext,arc_easy,hellaswag,winogrande,mmlu --skip-completed


In [ ]:
# 8. Phase D: Execute Generative Reasoning & Instruction Following (GSM8K & IFEval)
!python run_experiments.py --tasks gsm8k,ifeval --skip-completed


In [ ]:
# 9. Phase E & G: Run Statistical Analysis (10,000 Paired Bootstrap CIs, Spearman Proxy & Bit-Budget)
!python run_analysis.py --csv results/results_master.csv --bootstrap 10000

In [ ]:
# 10. Display Analytical Findings Inline
from IPython.display import Markdown, display
for rpt in ["results/degradation_report.md", "results/spearman_proxy_report.md", "results/bit_budget_guidelines.md"]:
    if os.path.exists(rpt):
        with open(rpt) as f:
            display(Markdown(f.read()))
        print("\n" + "="*75 + "\n")

In [ ]:
# 11. Archive All Results and Sample Logs for Download
!zip -r btp_evaluation_results.zip results/
print("✅ Results successfully zipped into btp_evaluation_results.zip!")